In [ ]:
# SMOTE BALANCING
# Target per kelas (konservatif):
#   0 - Normal      : 2500 (turun dari 2173 sedikit)
#   1 - Jantung     : 1500 (naik dari 914)
#   2 - Peny. Dalam : 2000 (turun dari 2698)  → undersample
#   3 - Paru-paru   : 1500 (naik dari 645)

import numpy as np
import pandas as pd
from collections import Counter
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline


In [ ]:
np.random.seed(42)

df = pd.read_csv('merge_DocLab_hybrid.csv')

FEATURES = ['JENIS_KELAMIN','UMUR_TAHUN','cholesterol total','creatinin',
            'fbs','rbs','hgb','lymfosit%','mch','mchc','mcv','ureum','wbc']
TARGET = 'NM_UNIT'

X = df[FEATURES]
y = df[TARGET]

print("Sebelum balancing:", Counter(y))

# Step 1: SMOTE dulu untuk kelas minoritas (1 dan 3)
# Step 2: Undersample kelas mayoritas (0 dan 2)
over  = SMOTE(sampling_strategy={1: 1500, 3: 1500}, random_state=42, k_neighbors=5)
under = RandomUnderSampler(sampling_strategy={0: 2100, 2: 2000}, random_state=42)

pipeline = ImbPipeline(steps=[('smote', over), ('under', under)])
X_res, y_res = pipeline.fit_resample(X, y)

print("Setelah balancing:", Counter(y_res))



Sebelum balancing: Counter({2: 2698, 0: 2173, 1: 914, 3: 645})
Setelah balancing: Counter({0: 2100, 2: 2000, 1: 1500, 3: 1500})


In [ ]:
# Rapikan tipe data
df_balanced = pd.DataFrame(X_res, columns=FEATURES)
df_balanced[TARGET] = y_res

df_balanced['JENIS_KELAMIN'] = df_balanced['JENIS_KELAMIN'].round().astype(int)
df_balanced['NM_UNIT']       = df_balanced['NM_UNIT'].astype(int)
df_balanced['UMUR_TAHUN']    = df_balanced['UMUR_TAHUN'].round(0)

lab_dec = {'cholesterol total':2,'creatinin':2,'fbs':1,'rbs':1,
           'hgb':1,'lymfosit%':1,'mch':1,'mchc':1,'mcv':1,'ureum':1,'wbc':1}
for col, dec in lab_dec.items():
    df_balanced[col] = df_balanced[col].round(dec)

print(f"\nShape final : {df_balanced.shape}")
print(f"Missing     : {df_balanced.isnull().sum().sum()}")
print(f"\nDistribusi final:")
label_map = {0:'Normal', 1:'Jantung', 2:'Peny.Dalam', 3:'Paru'}
for u, label in label_map.items():
    n = (df_balanced[TARGET]==u).sum()
    print(f"  {label:<12}: {n}")

OUTPUT_PATH = 'merge_DocLab_balanced.csv'
df_balanced.to_csv(OUTPUT_PATH, index=False)
print(f"\n💾 Disimpan: {OUTPUT_PATH}")


Shape final : (7100, 14)
Missing     : 0

Distribusi final:
  Normal      : 2100
  Jantung     : 1500
  Peny.Dalam  : 2000
  Paru        : 1500

💾 Disimpan: merge_DocLab_balanced.csv
